# SETUP

In [ ]:
CREATE_ANIMATION = False
CREATE_LABEL_FILES = False

In [ ]:
import os
import sys
import yaml
from nuscenes.nuscenes import NuScenes
import matplotlib.pyplot as plt
from IPython.display import display, HTML # Ensure HTML is imported for display

# Add project root to sys.path
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added project root to sys.path: {PROJECT_ROOT}")

# --- MODIFIED IMPORTS ---
from src.data_utils.label_generation import (
    generate_point_labels_for_scene,
    find_instances_in_scene # If needed directly by notebook, else it's internal to label_generation
)
from src.data_utils.nuscenes_helper import get_scene_sweep_data_sequence # New primary sweep getter

# Import animation functions from their new location
from src.visualization.animation_helpers import create_synchronized_animation

# Load config
try:
    config_path = os.path.join(PROJECT_ROOT, 'config/m_detector_config.yaml')
    if not os.path.exists(config_path) and os.path.basename(PROJECT_ROOT) == 'notebooks':
        # If notebook is in 'notebooks' subdir, try one level up for config
        config_path = os.path.join(os.path.dirname(PROJECT_ROOT), 'config/m_detector_config.yaml')

    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
except FileNotFoundError:
    print(f"Config file not found at {config_path}. Please check your path.")

# Initialize NuScenes
nusc = NuScenes(
    version=config['nuscenes']['version'],
    dataroot=config['nuscenes']['dataroot'],
    verbose=True
)

print("NuScenes and helper functions loaded.")

scene_index = 1 
my_scene_rec = nusc.scene[scene_index]
print(f"Selected Scene: {my_scene_rec['name']}")

# Create animation

In [ ]:
if CREATE_ANIMATION:
    all_sweep_data_for_animation = list(get_scene_sweep_data_sequence(nusc, my_scene_rec['token']))
    instances_tokens_for_anim = find_instances_in_scene(nusc, my_scene_rec['token'], min_annotations=1)

    plt.rcParams['animation.embed_limit'] = 2**128 # allow large animation
    print(f"Generating animation for {len(all_sweep_data_for_animation)} sweeps. Note, this may take a few minutes ...")
    animation_html, _ = create_synchronized_animation(
        nusc,
        instances_tokens_for_anim,
        all_sweep_data_for_animation, # Pass the new structure
        scene_name=my_scene_rec['name'],
        interval_ms=50, # Display interval (10 FPS for display)
        save_path=None, # Set to "output_animation.mp4" or similar to save
        # save_path="output_animation_refactored.mp4", # Example: to save
        save_writer="ffmpeg",
        save_fps=20, # Save at 20 FPS (native LiDAR frequency)
        save_dpi=200,
        point_downsample=10 # Downsample LiDAR points by factor of 2 for performance
    )
    
    if animation_html: display(animation_html)

# Generate label files

In [8]:
# Define the base directory where all scene label folders will be created
# This will create subfolders like "scene-xxxx/"
output_point_labels_base_dir = os.path.join(config["nuscenes"]["label_path"]) 
os.makedirs(output_point_labels_base_dir, exist_ok=True)
print(f"Base directory for point labels: {output_point_labels_base_dir}")

if CREATE_LABEL_FILES:
    # # --- Option 1: Generate point labels for the currently selected scene (my_scene) ---
    # print(f"\nGenerating point labels for scene: {my_scene['name']}")
    # generate_point_labels_for_scene(nusc, my_scene, output_point_labels_base_dir, verbose=True)
    # print("Point label generation for the selected scene complete.")


    # --- Option 2: Generate point labels for ALL scenes in the dataset ---
    # WARNING: This will be more computationally intensive and take more disk space!
    # For v1.0-mini, it's manageable. For the full dataset, it will be significant.

    print("\nStarting point label generation for ALL scenes...")
    for scene_idx in range(len(nusc.scene)):
        current_scene_record = nusc.scene[scene_idx]
        print(f"\n--- Processing scene {scene_idx + 1}/{len(nusc.scene)} for point labels: {current_scene_record['name']} ---")
        generate_point_labels_for_scene(nusc, current_scene_record, output_point_labels_base_dir, verbose=True)
    print("\nPoint label generation for ALL scenes complete.")
else:
    print("Skipping label file generation.")

Base directory for point labels: /home/drugge/staff-umbrella/TeamHolgerResearch/drugge/nuscenes_labels_interpolated

Starting point label generation for ALL scenes...

--- Processing scene 1/10 for point labels: scene-0061 ---
Processing scene for point labels: scene-0061 (cc8c0bf57f984915a77078b10eb33198)
Found 382 LiDAR sweeps for scene scene-0061.
Found 227 instances to process.
  Pre-calculating box states for instance 10/227 (e37421...)
  Pre-calculating box states for instance 20/227 (e1a406...)
  Pre-calculating box states for instance 30/227 (948655...)
  Pre-calculating box states for instance 40/227 (37ae87...)
  Pre-calculating box states for instance 50/227 (813d6c...)
  Pre-calculating box states for instance 60/227 (4a5a65...)
  Pre-calculating box states for instance 70/227 (3b54d4...)
  Pre-calculating box states for instance 80/227 (8fcc32...)
  Pre-calculating box states for instance 90/227 (f8fe56...)
  Pre-calculating box states for instance 100/227 (48d58b...)
  Pr

  Generating Labels: 100%|██████████| 382/382 [01:03<00:00,  5.98it/s]


Finished scene scene-0061. Generated 382 label files.

--- Processing scene 2/10 for point labels: scene-0103 ---
Processing scene for point labels: scene-0103 (fcbccedd61424f1b85dcbf8f897f9754)
Found 389 LiDAR sweeps for scene scene-0103.
Found 123 instances to process.
  Pre-calculating box states for instance 10/123 (1e4a74...)
  Pre-calculating box states for instance 20/123 (62cd17...)
  Pre-calculating box states for instance 30/123 (0bfc7d...)
  Pre-calculating box states for instance 40/123 (bfe685...)
  Pre-calculating box states for instance 50/123 (608f6c...)
  Pre-calculating box states for instance 60/123 (c56008...)
  Pre-calculating box states for instance 70/123 (b34125...)
  Pre-calculating box states for instance 80/123 (9439a2...)
  Pre-calculating box states for instance 90/123 (5d77c2...)
  Pre-calculating box states for instance 100/123 (713b6b...)
  Pre-calculating box states for instance 110/123 (fef3f0...)
  Pre-calculating box states for instance 120/123 (b9df

  Generating Labels: 100%|██████████| 389/389 [00:49<00:00,  7.92it/s]


Finished scene scene-0103. Generated 389 label files.

--- Processing scene 3/10 for point labels: scene-0553 ---
Processing scene for point labels: scene-0553 (6f83169d067343658251f72e1dd17dbc)
Found 398 LiDAR sweeps for scene scene-0553.
Found 57 instances to process.
  Pre-calculating box states for instance 10/57 (813f2a...)
  Pre-calculating box states for instance 20/57 (7a8b24...)
  Pre-calculating box states for instance 30/57 (e8ee78...)
  Pre-calculating box states for instance 40/57 (b2a522...)
  Pre-calculating box states for instance 50/57 (340ccb...)
All instance box states generated. Now creating point label files per sweep...


...

Finished scene scene-1077. Generated 400 label files.

--- Processing scene 9/10 for point labels: scene-1094 ---
Processing scene for point labels: scene-1094 (de7d80a1f5fb4c3e82ce8a4f213b450a)
Found 391 LiDAR sweeps for scene scene-1094.
Found 85 instances to process.
  Pre-calculating box states for instance 10/85 (a2cf4f...)
  Pre-calculating box states for instance 20/85 (a9822c...)
  Pre-calculating box states for instance 30/85 (dbdc55...)
  Pre-calculating box states for instance 40/85 (2bfbbe...)
  Pre-calculating box states for instance 50/85 (c530e5...)
  Pre-calculating box states for instance 60/85 (35f7d6...)
  Pre-calculating box states for instance 70/85 (995f67...)
  Pre-calculating box states for instance 80/85 (4d171d...)
All instance box states generated. Now creating point label files per sweep...


  Generating Labels: 100%|██████████| 391/391 [00:43<00:00,  8.93it/s]


Finished scene scene-1094. Generated 391 label files.

--- Processing scene 10/10 for point labels: scene-1100 ---
Processing scene for point labels: scene-1100 (e233467e827140efa4b42d2b4c435855)
Found 391 LiDAR sweeps for scene scene-1100.
Found 32 instances to process.
  Pre-calculating box states for instance 10/32 (be4d0d...)
  Pre-calculating box states for instance 20/32 (e3c44d...)
  Pre-calculating box states for instance 30/32 (1c6ea6...)
All instance box states generated. Now creating point label files per sweep...


  Generating Labels: 100%|██████████| 391/391 [00:37<00:00, 10.53it/s]

Finished scene scene-1100. Generated 391 label files.

Point label generation for ALL scenes complete.
